# Group Project - COMPAS Score

Data processing and model testing for the IAA Group Project.

- Gonçalo Ribau (119560)
- Igor Baltarejo (118832)
- Tiago Oliveira (118772)

## 0) Environment and Data setup

- Kernel: Python 3 (Anaconda)
- numpy, pandas, matplotlib

In [1]:
# Base imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional configuration
pd.set_option('display.max_columns', 100)

The raw data to be used in the project.

In [2]:
DATA_PATH = '../data/compas-scores-raw.csv'

df = pd.read_csv(DATA_PATH)
df.head()

,Person_ID,AssessmentID,Case_ID,Agency_Text,LastName,FirstName,MiddleName,Sex_Code_Text,Ethnic_Code_Text,DateOfBirth,ScaleSet_ID,ScaleSet,AssessmentReason,Language,LegalStatus,CustodyStatus,MaritalStatus,Screening_Date,RecSupervisionLevel,RecSupervisionLevelText,Scale_ID,DisplayText,RawScore,DecileScore,ScoreText,AssessmentType,IsCompleted,IsDeleted
0,50844,57167,51950,PRETRIAL,Fisher,Kevin,NaN,Male,Caucasian,12/05/92,22,Risk and Prescreen,Intake,English,Pretrial,Jail Inmate,Single,1/1/13 0:00,1,Low,7,Risk of Violence,-2.08,4,Low,New,1,0
1,50844,57167,51950,PRETRIAL,Fisher,Kevin,NaN,Male,Caucasian,12/05/92,22,Risk and Prescreen,Intake,English,Pretrial,Jail Inmate,Single,1/1/13 0:00,1,Low,8,Risk of Recidivism,-1.06,2,Low,New,1,0
2,50844,57167,51950,PRETRIAL,Fisher,Kevin,NaN,Male,Caucasian,12/05/92,22,Risk and Prescreen,Intake,English,Pretrial,Jail Inmate,Single,1/1/13 0:00,1,Low,18,Risk of Failure to Appear,15.00,1,Low,New,1,0
3,50848,57174,51956,PRETRIAL,KENDALL,KEVIN,NaN,Male,Caucasian,09/16/84,22,Risk and Prescreen,Intake,English,Pretrial,Jail Inmate,Married,1/1/13 0:00,1,Low,7,Risk of Violence,-2.84,2,Low,New,1,0
4,50848,57174,51956,PRETRIAL,KENDALL,KEVIN,NaN,Male,Caucasian,09/16/84,22,Risk and Prescreen,Intake,English,Pretrial,Jail Inmate,Married,1/1/13 0:00,1,Low,8,Risk of Recidivism,-1.50,1,Low,New,1,0


## 1) Initial Exploration

In [3]:
print('Shape:', df.shape)
print('Missing values:')
print(df.isna().sum().sort_values(ascending=False).head(15))

Shape: (60843, 28)
Missing values:
MiddleName          45219
ScoreText              45
Case_ID                 0
Person_ID               0
Agency_Text             0
LastName                0
FirstName               0
Sex_Code_Text           0
Ethnic_Code_Text        0
DateOfBirth             0
ScaleSet_ID             0
AssessmentID            0
ScaleSet                0
AssessmentReason        0
LegalStatus             0
dtype: int64


## 2) Preprocessing

In [ ]:
# Basic processing configuration
TARGET = "RawScore"
ASSESSMENT = "Risk of Recidivism"

df_clean = df.copy()
print(f"Initial rows: {len(df_clean):,}")

### 2.1 Filter Scope

Keep only **Risk of Recidivism** rows for a consistent modeling target.

In [ ]:
df_clean = df_clean[df_clean["DisplayText"] == ASSESSMENT].copy()
print(f"Rows after assessment filter ({ASSESSMENT}): {len(df_clean):,}")

### 2.2 Parse Dates and Build Age Feature

Convert date columns and compute age at screening.

In [ ]:
for col in ["Screening_Date", "DateOfBirth"]:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce", format="mixed")

df_clean["age_at_screening"] = (
    (df_clean["Screening_Date"] - df_clean["DateOfBirth"]).dt.days / 365.25
)

df_clean[["Screening_Date", "DateOfBirth", "age_at_screening"]].head()

### 2.3 Remove Invalid Ages and Missing Target

Apply basic quality rules before modeling.

In [ ]:
n_before_age = len(df_clean)
df_clean = df_clean[
    (df_clean["age_at_screening"].notna())
    & (df_clean["age_at_screening"] >= 10)
    & (df_clean["age_at_screening"] <= 100)
].copy()
print(f"Removed due to invalid age: {n_before_age - len(df_clean):,}")

n_before_target = len(df_clean)
df_clean = df_clean.dropna(subset=[TARGET]).copy()
print(f"Removed due to missing target ({TARGET}): {n_before_target - len(df_clean):,}")

### 2.4 Handle Missing Values and Final Check

Fill categorical and numerical missing values, then inspect the cleaned dataset.

In [ ]:
cat_cols = df_clean.select_dtypes(include=["object", "category"]).columns
num_cols = df_clean.select_dtypes(include=[np.number]).columns

df_clean[cat_cols] = df_clean[cat_cols].fillna("Unknown")
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Processed dataset summary")
print("Shape:", df_clean.shape)
print("Unique people:", df_clean["Person_ID"].nunique())
print(
    f"{TARGET} stats -> mean: {df_clean[TARGET].mean():.3f}, "
    f"std: {df_clean[TARGET].std():.3f}, "
    f"min: {df_clean[TARGET].min():.3f}, "
    f"max: {df_clean[TARGET].max():.3f}"
)

df_clean.head()

### Saving the data

Saving the data into a new csv file.

In [ ]:
OUTPUT_PATH = "../data/compas-scores-processed.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH, "| shape:", df_clean.shape)

## 3) Modeling

In [ ]:
# TODO: define X, y, train models, and compare metrics
# Placeholder example:
results = {}
results

## 4) Ethical Considerations and Limitations

- Bias risks
- Data limitations
- Impact of modeling decisions

## 5) Conclusions and Next Steps